In [1]:
# ============================================================
# CELL 1 — SETUP, LOAD DATA & BUILD BASE FEATURES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD ALL DATA
# ============================================================
results          = pd.read_csv('../data/results.csv')
races            = pd.read_csv('../data/races.csv')
drivers          = pd.read_csv('../data/drivers.csv')
constructors     = pd.read_csv('../data/constructors.csv')
pit_stops        = pd.read_csv('../data/pit_stops.csv')
qualifying       = pd.read_csv('../data/qualifying.csv')
driver_standings = pd.read_csv('../data/driver_standings.csv')
circuits         = pd.read_csv('../data/circuits.csv')

print("✅ All CSVs loaded.")

# ============================================================
# CLEAN DATA
# ============================================================
results.replace('\\N', np.nan, inplace=True)
qualifying.replace('\\N', np.nan, inplace=True)
driver_standings.replace('\\N', np.nan, inplace=True)

results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')
results['points']        = pd.to_numeric(results['points'],        errors='coerce')
results['grid']          = pd.to_numeric(results['grid'],          errors='coerce')
results['laps']          = pd.to_numeric(results['laps'],          errors='coerce')
results['position']      = pd.to_numeric(results['position'],      errors='coerce')

print("✅ Data cleaned.")

# ============================================================
# FILTER TO 2021-2024
# ============================================================
modern_races   = races[races['year'].between(2021, 2024)].copy()
modern_raceIds = modern_races['raceId'].tolist()
modern_results = results[results['raceId'].isin(modern_raceIds)].copy()

# Merge race info into results
modern_results = modern_results.merge(
    modern_races[['raceId', 'year', 'round', 'circuitId', 'name']],
    on='raceId', how='left'
)

print(f"\nDataset coverage:")
print(f"  Races   : {len(modern_races)}")
print(f"  Results : {len(modern_results)}")
print(f"  Years   : {sorted(modern_results['year'].unique())}")

# ============================================================
# TARGET VARIABLE
# ============================================================
modern_results['is_winner'] = (
    modern_results['positionOrder'] == 1
).astype(int)

total        = len(modern_results)
winners      = modern_results['is_winner'].sum()
non_winners  = total - winners
win_pct      = (winners / total) * 100

print(f"\nTarget variable — is_winner:")
print(f"  Winners (1)     : {winners}  ({win_pct:.1f}%)")
print(f"  Non-winners (0) : {non_winners}  ({100-win_pct:.1f}%)")
print(f"  Base rate       : 1 winner per race = ~5% positive class")

# ============================================================
# BASE FEATURE 1 — GRID POSITION
# ============================================================
modern_results['grid'] = modern_results['grid'].replace(0, 20)
modern_results['grid'] = modern_results['grid'].fillna(20)

# ============================================================
# BASE FEATURE 2 — DRIVER WIN RATE AT CIRCUIT
# ============================================================
circuit_wins = modern_results.groupby(
    ['driverId', 'circuitId']
).agg(
    races_at_circuit = ('raceId', 'count'),
    wins_at_circuit  = ('is_winner', 'sum')
).reset_index()

circuit_wins['win_rate_at_circuit'] = (
    circuit_wins['wins_at_circuit'] /
    circuit_wins['races_at_circuit']
)

modern_results = modern_results.merge(
    circuit_wins[['driverId', 'circuitId', 'win_rate_at_circuit']],
    on=['driverId', 'circuitId'], how='left'
)
modern_results['win_rate_at_circuit'] = (
    modern_results['win_rate_at_circuit'].fillna(0)
)

# ============================================================
# BASE FEATURE 3 — CONSTRUCTOR AVERAGE POINTS PER RACE
# ============================================================
constructor_avg = (
    modern_results.groupby('constructorId')['points']
    .mean()
    .reset_index()
)
constructor_avg.columns = ['constructorId', 'constructor_avg_points']

modern_results = modern_results.merge(
    constructor_avg, on='constructorId', how='left'
)
modern_results['constructor_avg_points'] = (
    modern_results['constructor_avg_points'].fillna(0)
)

# ============================================================
# BASE FEATURE 4 — DRIVER AVERAGE POINTS PER RACE
# ============================================================
driver_avg = (
    modern_results.groupby('driverId')['points']
    .mean()
    .reset_index()
)
driver_avg.columns = ['driverId', 'driver_avg_points']

modern_results = modern_results.merge(
    driver_avg, on='driverId', how='left'
)
modern_results['driver_avg_points'] = (
    modern_results['driver_avg_points'].fillna(0)
)

# ============================================================
# SUMMARY
# ============================================================
print(f"\nBase features built:")
print(f"  grid                    : qualifying / grid position")
print(f"  win_rate_at_circuit     : driver historical win % at this circuit")
print(f"  constructor_avg_points  : team average points per race")
print(f"  driver_avg_points       : driver average points per race")
print(f"\nFinal dataset shape : {modern_results.shape}")
print(f"\n✅ Cell 1 complete - Base features ready.")

✅ All CSVs loaded.
✅ Data cleaned.

Dataset coverage:
  Races   : 90
  Results : 1799
  Years   : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Target variable — is_winner:
  Winners (1)     : 90  (5.0%)
  Non-winners (0) : 1709  (95.0%)
  Base rate       : 1 winner per race = ~5% positive class

Base features built:
  grid                    : qualifying / grid position
  win_rate_at_circuit     : driver historical win % at this circuit
  constructor_avg_points  : team average points per race
  driver_avg_points       : driver average points per race

Final dataset shape : (1799, 26)

✅ Cell 1 complete - Base features ready.


In [2]:
# ============================================================
# CELL 2 — ADVANCED FEATURE ENGINEERING
# ============================================================

# ============================================================
# FEATURE 5 — DRIVER FORM (rolling avg points last 3 races)
# ============================================================
# Sort by driver, year, round so rolling is chronological
modern_results = modern_results.sort_values(
    ['driverId', 'year', 'round']
).reset_index(drop=True)

modern_results['driver_form_3'] = (
    modern_results.groupby('driverId')['points']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    .fillna(0)
)

print("✅ Feature 5 done - driver_form_3 (rolling 3 race avg points)")

# ============================================================
# FEATURE 6 — DRIVER FORM (rolling avg points last 5 races)
# ============================================================
modern_results['driver_form_5'] = (
    modern_results.groupby('driverId')['points']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .fillna(0)
)

print("✅ Feature 6 done - driver_form_5 (rolling 5 race avg points)")

# ============================================================
# FEATURE 7 — DRIVER RECENT WIN RATE (last 5 races)
# ============================================================
modern_results['recent_win_rate'] = (
    modern_results.groupby('driverId')['is_winner']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    .fillna(0)
)

print("✅ Feature 7 done - recent_win_rate (rolling 5 race win rate)")

# ============================================================
# FEATURE 8 — CONSTRUCTOR FORM (rolling avg points last 3 races)
# ============================================================
# Use sum of both drivers points per race for constructor form
constructor_race_pts = (
    modern_results.groupby(['constructorId', 'raceId'])['points']
    .sum()
    .reset_index()
)
constructor_race_pts.columns = ['constructorId', 'raceId',
                                 'constructor_race_total_pts']

modern_results = modern_results.merge(
    constructor_race_pts, on=['constructorId', 'raceId'], how='left'
)

modern_results = modern_results.sort_values(
    ['constructorId', 'year', 'round']
).reset_index(drop=True)

modern_results['constructor_form_3'] = (
    modern_results.groupby('constructorId')['constructor_race_total_pts']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    .fillna(0)
)

print("✅ Feature 8 done - constructor_form_3 (rolling 3 race team points)")

# ============================================================
# FEATURE 9 — GRID POSITION SQUARED
# (non-linear relationship — P1 advantage is exponentially bigger)
# ============================================================
modern_results['grid_squared'] = modern_results['grid'] ** 2

print("✅ Feature 9 done - grid_squared")

# ============================================================
# FEATURE 10 — IS FRONT ROW (grid 1 or 2)
# ============================================================
modern_results['is_front_row'] = (
    modern_results['grid'] <= 2
).astype(int)

print("✅ Feature 10 done - is_front_row")

# ============================================================
# FEATURE 11 — IS POLE POSITION (grid == 1)
# ============================================================
modern_results['is_pole'] = (
    modern_results['grid'] == 1
).astype(int)

print("✅ Feature 11 done - is_pole")

# ============================================================
# FEATURE 12 — DRIVER CHAMPIONSHIP POSITION AT THAT POINT
# ============================================================
driver_standings.replace('\\N', np.nan, inplace=True)
driver_standings['position'] = pd.to_numeric(
    driver_standings['position'], errors='coerce'
)
driver_standings['points'] = pd.to_numeric(
    driver_standings['points'], errors='coerce'
)

# Get standing position per driver per race
standings_per_race = driver_standings[
    ['raceId', 'driverId', 'position', 'points']
].copy()
standings_per_race.columns = [
    'raceId', 'driverId',
    'championship_position',
    'championship_points'
]

modern_results = modern_results.merge(
    standings_per_race, on=['raceId', 'driverId'], how='left'
)
modern_results['championship_position'] = (
    modern_results['championship_position'].fillna(20)
)
modern_results['championship_points'] = (
    modern_results['championship_points'].fillna(0)
)

print("✅ Feature 12 done - championship_position + championship_points")

# ============================================================
# FEATURE 13 — DNF RATE (driver reliability)
# ============================================================
# statusId 1 = Finished, everything else = DNF
results_all = pd.read_csv('../data/results.csv')
results_all.replace('\\N', np.nan, inplace=True)
results_all['statusId'] = pd.to_numeric(
    results_all['statusId'], errors='coerce'
)

dnf_rate = results_all.groupby('driverId').apply(
    lambda x: (x['statusId'] != 1).sum() / len(x)
).reset_index()
dnf_rate.columns = ['driverId', 'dnf_rate']

modern_results = modern_results.merge(
    dnf_rate, on='driverId', how='left'
)
modern_results['dnf_rate'] = modern_results['dnf_rate'].fillna(0)

print("✅ Feature 13 done - dnf_rate (driver reliability score)")

# ============================================================
# FEATURE 14 — ROUND NUMBER (race number in season)
# (early season vs late season dynamics)
# ============================================================
# Already in modern_results as 'round'
print("✅ Feature 14 done - round (already in dataset)")

# ============================================================
# FINAL FEATURE SET SUMMARY
# ============================================================
feature_cols = [
    'grid',
    'grid_squared',
    'is_pole',
    'is_front_row',
    'driver_avg_points',
    'driver_form_3',
    'driver_form_5',
    'recent_win_rate',
    'win_rate_at_circuit',
    'constructor_avg_points',
    'constructor_form_3',
    'championship_position',
    'championship_points',
    'dnf_rate',
    'round',
]

print(f"\n{'='*50}")
print(f"  FINAL FEATURE SET ({len(feature_cols)} features)")
print(f"{'='*50}")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:>2}. {f}")

# Check for nulls in features
null_counts = modern_results[feature_cols].isnull().sum()
if null_counts.sum() == 0:
    print(f"\n✅ No nulls in any feature column.")
else:
    print(f"\n⚠️  Nulls found:")
    print(null_counts[null_counts > 0])

print(f"\nFinal dataset shape : {modern_results.shape}")
print(f"\n✅ Cell 2 complete - All {len(feature_cols)} features ready.")

✅ Feature 5 done - driver_form_3 (rolling 3 race avg points)
✅ Feature 6 done - driver_form_5 (rolling 5 race avg points)
✅ Feature 7 done - recent_win_rate (rolling 5 race win rate)
✅ Feature 8 done - constructor_form_3 (rolling 3 race team points)
✅ Feature 9 done - grid_squared
✅ Feature 10 done - is_front_row
✅ Feature 11 done - is_pole
✅ Feature 12 done - championship_position + championship_points
✅ Feature 13 done - dnf_rate (driver reliability score)
✅ Feature 14 done - round (already in dataset)

  FINAL FEATURE SET (15 features)
   1. grid
   2. grid_squared
   3. is_pole
   4. is_front_row
   5. driver_avg_points
   6. driver_form_3
   7. driver_form_5
   8. recent_win_rate
   9. win_rate_at_circuit
  10. constructor_avg_points
  11. constructor_form_3
  12. championship_position
  13. championship_points
  14. dnf_rate
  15. round

✅ No nulls in any feature column.

Final dataset shape : (1799, 37)

✅ Cell 2 complete - All 15 features ready.
